# 고급 RAG 기법 — Ensemble + MMR으로 검색 품질 향상

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 기본 RAG의 한계(벡터 검색만 사용할 때의 문제)를 명확히 설명하기
- **EnsembleRetriever**로 BM25와 FAISS를 결합하는 하이브리드 검색 구현하기
- **MMR(Maximum Marginal Relevance)**로 다양성 있는 검색 결과 얻기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인
- 6-5 섹션의 EnsembleRetriever와 MMR 개념

---

```mermaid
flowchart TD
    Q[사용자 질문]:::input --> B[BM25<br/>키워드 검색]:::process
    Q --> VM[FAISS + MMR<br/>의미+다양성 검색]:::process
    B -- 가중치 0.4 --> E[Ensemble Retriever<br/>RRF 결합]:::process
    VM -- 가중치 0.6 --> E
    E --> P[Prompt 생성]:::process
    P --> L[LLM 답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 기본 RAG의 한계

| 문제 | 예시 |
|------|------|
| 키워드 미매칭 | "AI"로 검색했는데 문서엔 "인공지능"만 있는 경우 |
| 중복 결과 | 유사한 내용의 문서가 반복적으로 검색됨 |
| 동의어 처리 | "LLM" = "대규모 언어 모델" 연결 불가 |

> **실무 팁**: Ensemble의 가중치 `[BM25, FAISS]`를 `[0.3, 0.7]`처럼 의미 검색에 더 높은 가중치를 주면 자연어 질문에 더 잘 응답해요.

## 기본 RAG의 한계

### 1. 벡터 검색만 사용할 때의 문제

- **의미는 비슷하지만 정확한 키워드가 없으면 검색 실패**
  - 예: "AI"를 검색했는데 문서에는 "인공지능"만 있는 경우
- **빈도가 낮은 중요 키워드 놓침**
  - 예: 고유명사나 기술 용어

### 2. 검색 결과의 중복

- 유사한 내용의 문서들이 반복적으로 검색됨
- 다양한 관점의 정보 부족

### 해결책

- **Ensemble Retriever**: Dense(벡터) + Sparse(키워드) 검색 결합
- **MMR**: 관련성과 다양성을 동시에 고려

## 환경 설정

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 1. Ensemble Retriever

### 개념

Ensemble Retriever는 여러 검색 방식을 결합하여 더 나은 검색 결과를 얻습니다.

```
                  User Query
                      |
         +------------+------------+
         |                         |
    BM25 검색               벡터 검색
  (키워드 기반)           (의미 기반)
         |                         |
         +------------+------------+
                      |
              Ensemble Retriever
            (가중치로 결합)
                      |
                최종 검색 결과
```

### BM25 vs 벡터 검색

| 특징 | BM25 (Sparse) | 벡터 검색 (Dense) |
|------|---------------|------------------|
| 방식 | 키워드 매칭 | 의미 유사도 |
| 장점 | 정확한 용어 검색 | 동의어, 유사 개념 검색 |
| 단점 | 의미 파악 불가 | 정확한 키워드 놓칠 수 있음 |
| 예시 | "GPT-4" 정확히 검색 | "대형 언어 모델" ≈ "LLM" |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

### 문서 준비

In [ ]:
# 문서 로드 및 분할
loader = PyMuPDFLoader("../6-1_DocumentLoaders/data/sample-rag-brief.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
split_documents = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(split_documents)}")

### Ensemble Retriever 생성

In [ ]:
# ---------------------------------------------------
# BM25 + FAISS Ensemble Retriever 구성
# ---------------------------------------------------
# ============================================================
# TODO: BM25Retriever와 FAISS Retriever를 만들고 EnsembleRetriever로 결합하세요
# 힌트: EnsembleRetriever(retrievers=[bm25, faiss], weights=[0.5, 0.5])
# 예상 결과: "Ensemble Retriever 생성 완료" 출력
# ============================================================


### 검색 비교 테스트

In [ ]:
# ---------------------------------------------------
# BM25 vs FAISS vs Ensemble 검색 결과 비교
# ---------------------------------------------------
# ============================================================
# TODO: 동일한 쿼리로 BM25, FAISS, Ensemble 각각의 검색 결과를 비교하세요
# 힌트: bm25_retriever.invoke(query), faiss_retriever.invoke(query), ensemble_retriever.invoke(query)
# 예상 결과: 세 가지 검색기의 결과가 각각 출력됨
# ============================================================


## 2. MMR (Maximum Marginal Relevance)

### 개념

MMR은 검색 결과의 **관련성(Relevance)**과 **다양성(Diversity)**을 동시에 고려합니다.

**문제 상황**:
```
질문: "인공지능의 활용 분야는?"

일반 검색 결과:
1. 의료 분야 AI 활용
2. 의료 분야 AI 진단
3. 의료 분야 AI 치료  ← 모두 의료 분야만!
4. 의료 AI 사례
```

**MMR 검색 결과**:
```
1. 의료 분야 AI 활용
2. 금융 분야 AI 활용  ← 다양한 분야
3. 제조업 AI 활용
4. 교육 AI 활용
```

### 작동 원리

MMR 점수 = λ × (관련성) - (1-λ) × (중복도)

- λ = 1: 관련성만 고려 (일반 검색)
- λ = 0: 다양성만 고려
- λ = 0.5: 균형

In [ ]:
# ---------------------------------------------------
# MMR Retriever 생성 — 관련성·다양성 균형 검색
# ---------------------------------------------------
# ============================================================
# TODO: as_retriever(search_type="mmr")로 MMR 검색기를 만드세요
# 힌트: fetch_k=20 (후보), k=4 (최종), lambda_mult=0.5 (균형)
# 예상 결과: "MMR Retriever 생성 완료" 출력
# ============================================================


### 일반 검색 vs MMR 비교

In [ ]:
# ---------------------------------------------------
# 일반 유사도 검색 vs MMR 검색 비교
# ---------------------------------------------------
# ============================================================
# TODO: 동일한 쿼리로 일반 검색과 MMR 검색 결과를 비교하세요
# 힌트: faiss_retriever.invoke(query) vs mmr_retriever.invoke(query)
# 예상 결과: 일반 검색 결과와 MMR 검색 결과(다양성 고려)가 각각 출력됨
# ============================================================


## 3. Ensemble + MMR 결합

최고의 성능을 위해 Ensemble Retriever와 MMR을 함께 사용할 수 있습니다.

In [ ]:
# ---------------------------------------------------
# Ensemble + MMR 결합 — 최고 성능 검색기 구성
# ---------------------------------------------------
# ============================================================
# TODO: MMR을 적용한 FAISS Retriever와 BM25를 EnsembleRetriever로 결합하세요
# 힌트: weights=[0.4, 0.6]으로 의미 검색에 약간 더 가중치 부여
# 예상 결과: "Ensemble + MMR Retriever 생성 완료" 출력
# ============================================================


## 4. 고급 RAG 체인 구축

In [ ]:
# ---------------------------------------------------
# 고급 RAG 체인 조립 — Ensemble+MMR 검색기 연결
# ---------------------------------------------------
# ============================================================
# TODO: ensemble_mmr_retriever, prompt, llm을 LCEL로 연결해 RAG 체인을 만드세요
# 힌트: {"context": ensemble_mmr_retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
# 예상 결과: "고급 RAG 체인 생성 완료" 출력
# ============================================================


### 성능 비교: 기본 vs 고급 RAG

In [ ]:
# ---------------------------------------------------
# 기본 RAG vs 고급 RAG 답변 품질 비교
# ---------------------------------------------------
# ============================================================
# TODO: 동일한 질문을 basic_rag_chain과 advanced_rag_chain에 각각 invoke하세요
# 힌트: rag_chain.invoke(test_question)으로 두 체인의 답변을 비교
# 예상 결과: 고급 RAG가 더 다양하고 풍부한 답변을 생성함
# ============================================================


## 💡 핵심 정리

### Ensemble Retriever

**장점**:
- ✅ 키워드와 의미 검색의 장점 결합
- ✅ 정확한 용어와 유사 개념 모두 검색
- ✅ 검색 정확도 향상

**사용 시나리오**:
- 전문 용어가 많은 문서 (의료, 법률, 기술)
- 고유명사 검색이 중요한 경우

### MMR (Maximum Marginal Relevance)

**장점**:
- ✅ 검색 결과의 다양성 확보
- ✅ 중복 내용 제거
- ✅ 다양한 관점의 정보 제공

**사용 시나리오**:
- 포괄적인 정보가 필요한 경우
- 여러 관점의 답변이 필요한 경우

### 파라미터 튜닝 가이드

**Ensemble 가중치**:
```python
weights=[0.7, 0.3]  # 키워드 중심 (전문 용어 많을 때)
weights=[0.5, 0.5]  # 균형 (일반적)
weights=[0.3, 0.7]  # 의미 중심 (동의어 많을 때)
```

**MMR lambda_mult**:
```python
lambda_mult=1.0  # 관련성만 고려 (일반 검색과 동일)
lambda_mult=0.7  # 관련성 우선
lambda_mult=0.5  # 균형 (권장)
lambda_mult=0.3  # 다양성 우선
```

### 성능 향상 효과

| 기법 | 검색 정확도 | 다양성 | 처리 시간 |
|------|-----------|--------|----------|
| 기본 RAG | ⭐⭐⭐ | ⭐⭐ | 빠름 |
| + Ensemble | ⭐⭐⭐⭐ | ⭐⭐ | 보통 |
| + MMR | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | 약간 느림 |
| Ensemble + MMR | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | 보통 |

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 구성 | BM25 (Sparse) + FAISS with MMR (Dense) → EnsembleRetriever |
| MMR의 역할 | 유사도가 높지만 중복되지 않는 다양한 문서 선택 |
| BM25의 역할 | 키워드 정확 매칭으로 의미 검색이 놓치는 문서 보완 |
| Ensemble 가중치 | `[0.5, 0.5]` 기본 — 키워드 중심 도메인은 BM25 가중치를 높여요 |
| 응용 효과 | 기본 RAG 대비 동일 쿼리에서 더 다양하고 보완적인 문서 검색 |

### 고급 RAG 구성 전략

| 전략 | 구성 | 적합한 상황 |
|------|------|-------------|
| 기본 | FAISS Similarity | 단순 의미 검색으로 충분할 때 |
| 다양성 | FAISS MMR | 비슷한 문서 중복을 피하고 싶을 때 |
| 키워드 보완 | BM25 + FAISS Ensemble | 전문 용어, 고유명사가 많을 때 |
| **고급 (이 노트북)** | **BM25 + FAISS(MMR) Ensemble** | **다양성과 키워드 보완 모두 필요할 때** |

### 다음 노트북 예고

**03-Conversation-RAG** — 이전 대화 내용을 기억하면서 후속 질문을 처리하는 대화형 RAG를 배워요. `RunnableWithMessageHistory`와 세션 관리로 멀티턴 QA 시스템을 구축해요.